# 03 — Model experiments

Comparamos baselines vs. HGB. El código vive en `src/`; aquí lo importamos y narramos resultados.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from retail_ops_forecasting.config import load_config
from retail_ops_forecasting.data import load_transactions, load_stores, load_calendar, build_merged
sns.set_style('whitegrid')
cfg = load_config(ROOT / 'configs' / 'config.yaml')

from retail_ops_forecasting import baselines, modeling, features, splits, metrics
from retail_ops_forecasting.config import load_model_config
mcfg = load_model_config(ROOT / 'configs' / 'model_config.yaml')

## Cargar feature matrix de demand

In [2]:
p = ROOT / 'data' / 'processed' / 'demand_features.csv'
if (p.with_suffix('.parquet')).exists():
    feat = pd.read_parquet(p.with_suffix('.parquet'))
else:
    feat = pd.read_csv(p, parse_dates=['date'])
feat.shape

(203958, 53)

## Split temporal

In [3]:
s = splits.time_based_split(feat, cfg.splits)
print('train:', len(s.train), 'val:', len(s.val), 'test:', len(s.test))

train: 131016 val: 29280 test: 43662


## Baseline lag-7 — métricas val/test

In [4]:
target = cfg.targets.demand_primary
feat['lag7'] = baselines.naive_lag(feat, target, lag=7, entity_cols=['store_id','category'])
for name, idx in [('val', s.val), ('test', s.test)]:
    sub = feat.loc[idx].dropna(subset=[target,'lag7'])
    m = metrics.summarize(sub[target], sub['lag7'])
    print(name, {k: round(v,4) if isinstance(v,float) else v for k,v in m.items()})

val {'wape': 0.4186, 'mae': 313.328, 'rmse': 744.3097, 'smape': 0.3168, 'bias': 15.2325, 'n': 29280}
test {'wape': 0.3229, 'mae': 228.6949, 'rmse': 548.0485, 'smape': 0.2873, 'bias': -16.1504, 'n': 43662}


## Cargar resultados experimentales (después de `make train-demand`)

In [5]:
import json
p = ROOT / 'reports' / 'demand_experiment_summary.json'
if p.exists():
    sm = json.loads(p.read_text())
    rows = []
    for run, sp in sm.items():
        for split, m in sp.items():
            row = {'run': run, 'split': split, **{k:v for k,v in m.items() if isinstance(v,(int,float))}}
            rows.append(row)
    pd.DataFrame(rows).round(4)
else:
    print('Run `make train-demand` first.')